In [3]:
import pandas as pd

# 1. Carga y limpieza inicial


In [ ]:
df = pd.read_excel('E:/osley/Barecelona activa/curso especializacion analisis de datos/sprint 8/sprint10_complex.xlsx', header=3)

In [24]:
df.info()

<class 'pandas.DataFrame'>
Index: 928 entries, 0 to 1004
Data columns (total 20 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Nom                         928 non-null    str           
 1   Cognoms                     928 non-null    str           
 2   DNI                         928 non-null    str           
 3   País d'origen               928 non-null    str           
 4   Ciutat                      928 non-null    str           
 5   Dia de Naixement            928 non-null    int64         
 6   Mes de Naixement            928 non-null    int64         
 7   Any de Naixement            928 non-null    int64         
 8   Gènere                      926 non-null    str           
 9   Salari mensual              894 non-null    float64       
 10  Fills                       392 non-null    object        
 11  No Fills                    539 non-null    object        
 12  Grup Prof

In [12]:
arreglo_nombres = df["Carrec"].drop_duplicates().to_numpy()
print(arreglo_nombres)



['Cap de projecte' 'Senior analyst' 'Tecnic IT' 'Data Analyst'
 'Administratiu' 'Responsable de vendes' 'Analista junior' 'Project lead'
 'Admin.' 'Analyst' 'Analista' 'Data analyst' 'Cap de Projecte' nan]


In [25]:
df['Carrec'].unique()

<StringArray>
[      'Cap de projecte',        'Senior analyst',             'Tecnic IT',
          'Data Analyst',         'Administratiu', 'Responsable de vendes',
       'Analista junior',          'Project lead',                'Admin.',
               'Analyst',              'Analista',          'Data analyst',
       'Cap de Projecte',                     nan]
Length: 14, dtype: str

In [7]:
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

In [8]:
df.dropna(how='all', inplace=True)
df = df.drop(1006)

In [13]:
df.rename(columns={'CÃ\xa0rrec': 'Carrec'}, inplace=True)
df.dropna(how='all', inplace=True)

In [14]:
# 2. Función de limpieza de texto optimizada
def corregir_subcadenas_campo(dataframe, columnas_a_corregir, lista_mal, lista_bien):
    cambios = dict(zip(lista_mal, lista_bien))
    for col in columnas_a_corregir:
        if col in dataframe.columns:
            dataframe[col] = dataframe[col].astype(str)
            for mal, bien in cambios.items():
                dataframe[col] = dataframe[col].str.replace(mal, bien, regex=False)


errores = ['Ã\xa0', 'Ã©', 'Ã\xad','Ãº','Ã‰','Ã±','Ã³','Ã¡','Ã¨','Ã\x81','Ã¤','Ã¼','Ã¶','Ã\xa0','Ã§','Ã¸' ]
correcciones = ['à', 'é', 'í','ú','Á','ñ','ó','á','e','A','ä','ü','ö','a','ci','ø' ]


lista_campos = ['Nom', 'Cognoms', 'Ciutat', 'Carrec',"País d'origen"]
corregir_subcadenas_campo(df, lista_campos, errores, correcciones)

In [ ]:
# 3. Homologación de Género mediante Conjuntos Compactos
df['Gènere'] = df['Gènere'].replace({'H': 'Home', 'D': 'Dona', 'F': 'Dona'})

nombres_dona = {
    'Adri', 'Aina', 'Alba', 'Alexia', 'Anna', 'Camille', 'Carmen', 'Charlotte', 
    'Chiara', 'Clara', 'Claudia', 'Francesca', 'Elena', 'Élise', 'Irene', 'Gabriele', 
    'Giulia', 'Joana', 'Hannah', 'Ilaria', 'Marie', 'Marta', 'Marina', 'Martina', 
    'Mia', 'Katharina', 'Laia', 'Nuria', 'Sara', 'Sofie', 'Simone', 'Valentina', 
    'Lucía', 'Léa'
}

nombres_home = {
    'Alejandro', 'Alessandro', 'Antoine', 'Ben', 'Carlos', 'David', 'Adrià', 
    'Xavier', 'Erik', 'Felix', 'Hugo', 'Elias', 'Lucas', 'Lars', 'Jordi', 'Marc', 
    'Magnus', 'Marco', 'Nil', 'Noa', 'Matteo', 'Max', 'Oriol', 'Paolo', 'Pol', 
    'Noah', 'Pau', 'Paul', 'Nicolas', 'Riccardo', 'Sergio', 'Thomas', 'Víctor', 
    'Luca', 'Guillem'
}

# Compresión en un solo diccionario ejecutable
correccion_generos = {**{n: 'Dona' for n in nombres_dona}, **{n: 'Home' for n in nombres_home}}
df['Gènere'] = df['Nom'].map(correccion_generos).combine_first(df['Gènere'])

In [ ]:
# 4. Corrección Automatizada de Fechas
df['Any de Naixement'] = pd.to_numeric(df['Any de Naixement'], errors='coerce').fillna(0).astype(int)
df['Mes de Naixement'] = pd.to_numeric(df['Mes de Naixement'], errors='coerce').fillna(0).astype(int)
df['Dia de Naixement'] = pd.to_numeric(df['Dia de Naixement'], errors='coerce').fillna(0).astype(int)

df['Fecha_Nacimiento_Validada'] = pd.to_datetime(
    df['Any de Naixement'].astype(str) + '-' + df['Mes de Naixement'].astype(str) + '-' + df['Dia de Naixement'].astype(str),
    format='%Y-%m-%d', 
    errors='coerce'
)

df.dropna(subset=['Fecha_Nacimiento_Validada'], inplace=True)

In [117]:
columnas_a_eliminar = ["Any de Naixement", "Mes de Naixement", "Dia de Naixement"]
df = df.drop(columns=columnas_a_eliminar, errors="ignore")

In [ ]:
# 5. Arreglar columna 'Salari mensual'
df['Salari mensual'] = df['Salari mensual'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
df['Salari mensual'] = pd.to_numeric(df['Salari mensual'], errors='coerce').round(2)

In [20]:
# 6. Análisis de Duplicados
df_todos_duplicados = df[df.duplicated(subset=['DNI'], keep=False)]
df_duplicados_agrupados = df_todos_duplicados.sort_values(by='DNI')

print(f"Se encontraron {len(df_duplicados_agrupados)} filas asociadas a DNIs duplicados:\n")
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    display(df_duplicados_agrupados)


Se encontraron 24 filas asociadas a DNIs duplicados:



,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,Carrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat,Fecha_Nacimiento_Validada
604,Laia,Navarro,15909601K,Espanya,Bilbao,30,5,1958,Dona,1676.00,True,False,Grup A,Analista,1.0,True,50.0,NaN,14.2,1958-05-30
250,Laia,Navarro Vila,15909601K,Espanya,Bilbao,30,5,1958,Dona,1235.00,True,True,Grup D,Analista junior,0.0,False,NaN,NaN,10.3,1958-05-30
759,Oriol,Gómez,16618423V,Espanya,Sevilla,10,1,1968,Home,2000.00,NaN,NaN,Grup A,Responsable de vendes,5.0,NaN,NaN,NaN,20.7,1968-01-10
758,Oriol,Gómez,16618423V,Espanya,Malaga,10,1,1968,Home,1879.00,True,NaN,Grup B,Data Analyst,4.0,False,NaN,NaN,NaN,1968-01-10
754,Oriol,Gómez,16618423V,Espanya,Barcelona,10,1,1968,Home,2.36,True,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,1968-01-10
654,Oriol,Gómez,16618423V,Espanya,Madrid,10,1,1968,Home,1731.00,True,NaN,Grup B,Data analyst,4.0,NaN,NaN,NaN,NaN,1968-01-10
692,Alexia,Pérez Díaz,49529438G,Espanya,Malaga,12,10,1981,Dona,1663.00,NaN,True,Grup C,Cap de projecte,NaN,True,80000.0,10.7,16.9,1981-10-12
558,Alexia,Pérez Díaz,49529438G,Espanya,Malaga,12,10,1981,Dona,1176.00,NaN,True,Grup A,Data Analyst,0.0,False,NaN,NaN,38.0,1981-10-12
757,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15,7,1973,Home,786.00,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9,1973-07-15
1004,Oriol,Ferrer,52590001R,Espanya,Zaragoza,15,7,1973,Home,786.00,NaN,True,Grup A,Analista,0.0,True,81381.0,6.5,13.9,1973-07-15


Eliminacion de registros duplicados

Muestro solo los registros con un mismo DNI para poder analizar mejor.

In [ ]:
df_resultado = df[df['DNI'] == '15909601K']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

      Nom       Cognoms        DNI País d'origen  Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere  Salari mensual Fills No Fills Grup Professional           Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat Fecha_Nacimiento_Validada
250  Laia  Navarro Vila  15909601K       Espanya  Bilbao                30                 5              1958   Dona          1235.0  True     True            Grup D  Analista junior           0.0    False        NaN                   NaN                        10.3                1958-05-30
604  Laia       Navarro  15909601K       Espanya  Bilbao                30                 5              1958   Dona          1676.0  True    False            Grup A         Analista           1.0     True       50.0                   NaN                        14.2                1958-05-30


Con e DNI: 15909601K , 
- eliminar el registro 250 porque en es la misma persona que ha tenido hijo y ha mejorado en el cargo, ademas adquirio coche

In [119]:
df = df.drop(250)

In [ ]:
df_resultado = df[df['DNI'] == '16618423V']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

       Nom Cognoms        DNI País d'origen     Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere  Salari mensual Fills No Fills Grup Professional                 Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat Fecha_Nacimiento_Validada
654  Oriol   Gómez  16618423V       Espanya     Madrid                10                 1              1968   Home         1731.00  True      NaN            Grup B           Data analyst           4.0      NaN        NaN                   NaN                         NaN                1968-01-10
754  Oriol   Gómez  16618423V       Espanya  Barcelona                10                 1              1968   Home            2.36  True      NaN               NaN                    NaN           4.0      NaN        NaN                   NaN                         NaN                1968-01-10
758  Oriol   Gómez  16618423V       Espanya     Malaga                10                 1              19

Con e DNI: 16618423V , eliminar el registro 
- Eliminamos los registros 654,754,,758, al ser la misma persona que ha cambiado de ciudad, ha tenido un hijo mas y cambiado de trabajo, por loq ue queda el registro 759 

In [121]:
df = df.drop([654,754,758])

In [ ]:
df_resultado = df[df['DNI'] == '49529438G']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

        Nom     Cognoms        DNI País d'origen  Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere  Salari mensual Fills No Fills Grup Professional           Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat Fecha_Nacimiento_Validada
558  Alexia  Pérez Díaz  49529438G       Espanya  Malaga                12                10              1981   Dona          1176.0   NaN     True            Grup A     Data Analyst           0.0    False        NaN                   NaN                        38.0                1981-10-12
692  Alexia  Pérez Díaz  49529438G       Espanya  Malaga                12                10              1981   Dona          1663.0   NaN     True            Grup C  Cap de projecte           NaN     True    80000.0                  10.7                        16.9                1981-10-12


Con e DNI: 49529438G , 
- eliminar el registro 558 porque es la mims persona que ha cambiado de cargo y ha comprado un coche

In [122]:
df = df.drop(558)

In [ ]:
df_resultado = df[df['DNI'] == '52590001R']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

        Nom Cognoms        DNI País d'origen    Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere  Salari mensual Fills No Fills Grup Professional    Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat Fecha_Nacimiento_Validada
757   Oriol  Ferrer  52590001R       Espanya  Zaragoza                15                 7              1973   Home           786.0   NaN     True            Grup A  Analista           0.0     True    81381.0                   6.5                        13.9                1973-07-15
1004  Oriol  Ferrer  52590001R       Espanya  Zaragoza                15                 7              1973   Home           786.0   NaN     True            Grup A  Analista           0.0     True    81381.0                   6.5                        13.9                1973-07-15


Con e DNI: 52590001R
- eliminar el registro 1004 pues los dos registros son identicos por lo que no parta nada nuevo

In [123]:
df = df.drop(1004)

In [ ]:
df_resultado = df[df['DNI'] == '54013571W']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

         Nom      Cognoms        DNI País d'origen    Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere  Salari mensual Fills No Fills Grup Professional        Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat Fecha_Nacimiento_Validada
89   Guillem  Pujol Costa  54013571W       Espanya  A Coruña                17                 7              1983   Home            1.57  True      NaN            Grup B  Project lead           4.0    False        NaN                   NaN                        14.6                1983-07-17
142    Marta         Ruiz  54013571W       Espanya     Palma                24                 5              1982   Dona         1276.00  True     True            Grup A      Analista           0.0    False        NaN                   NaN                        15.7                1982-05-24


Con e DNI: 54013571W
- eliminar el registro 89 porque no tiene un salario coherente.

In [124]:
df = df.drop(89)

In [ ]:
df_resultado = df[df['DNI'] == '55283701C']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

    Nom          Cognoms        DNI País d'origen Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere  Salari mensual Fills No Fills Grup Professional    Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat Fecha_Nacimiento_Validada
36  Ida  Karlsen Eriksen  55283701C       Noruega   Oslo                 2                 9              1992   Dona          1614.0  True      NaN            Grup B  Analista           3.0      NaN        NaN                   NaN                        10.6                1992-09-02
40  Ida  Karlsen Eriksen  55283701C       Noruega   Oslo                 2                 9              1992   Dona          1614.0  True      NaN            Grup B  Analista           3.0      NaN        NaN                   NaN                        10.6                1992-09-02


Con e DNI: 55283701C
- se puede eliminar cualquiera de los dos registros pue son identicos.eliminemos el registro 40

In [125]:
df = df.drop(40)

In [23]:
df_resultado = df[df['DNI'] == '55297384F']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

          Nom     Cognoms        DNI País d'origen    Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere  Salari mensual Fills No Fills Grup Professional                 Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat Fecha_Nacimiento_Validada
730   Claudia  Gil Ferrer  55297384F       Espanya  Valencia                 1                11              1981   Dona            1.79   NaN     True            Grup B  Responsable de vendes           0.0     True    29952.0                   9.2                        20.3                1981-11-01
1003  Claudia  Gil Ferrer  55297384F       Espanya  Valencia                 1                11              1981   Dona            1.79   NaN     True            Grup B  Responsable de vendes           0.0     True    29952.0                   9.2                        20.3                1981-11-01


Con e DNI: 55297384F
- se puede eliminar cualquiera de los dos registros pues son identicos.

In [126]:
df = df.drop(1003)

In [26]:
df_resultado = df[df['DNI'] == '90585882A']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

         Nom      Cognoms        DNI País d'origen     Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere  Salari mensual Fills No Fills Grup Professional                 Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat Fecha_Nacimiento_Validada
25      Sara        Serra  90585882A       Espanya  Tarragona                28                 5              2002   Dona           993.0   NaN     True            Grup A          Administratiu           0.0    False        NaN                   NaN                        15.8                2002-05-28
104    Oriol  Vila Casals  90585882A       Espanya     Madrid                30                 9              1964   Home          1987.0   NaN     True            Grup B  Responsable de vendes           0.0    False        NaN                   NaN                        13.5                1964-09-30
114      Pol        Serra  90585882A       Espanya     Girona                 4      

Con e DNI: 90585882A
- se puede eliminar los registros 25,913 y dejar el registro 1000 poque es la misma persona y lo que se observa es un aumento de salario
-eliminaria los registro 281 y 114 porque hay valores refrenete a los hijos que estan en Nan, y el registro 104 porque si hay un evolucion en el salario de Sara Serra es porque ella esla que le pertenece ese DNI.

In [127]:
df = df.drop([25,913,281,114])

In [34]:
df_resultado = df[df['DNI'] == '97954019T']

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    print(df_resultado)

       Nom Cognoms        DNI País d'origen  Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere  Salari mensual Fills No Fills Grup Professional                 Carrec  Nombre_fills Te_cotxe  Km_anuals  Consum_mitja_L_100km  Temperatura_mitjana_ciutat Fecha_Nacimiento_Validada
372   Alba   Muñoz  97954019T       Espanya  Bilbao                25                10              1983   Dona          1409.0   NaN     True            Grup B  Responsable de vendes           0.0      NaN        NaN                   NaN                        14.7                1983-10-25
1002  Alba   Muñoz  97954019T       Espanya  Bilbao                25                10              1983   Dona          1409.0   NaN     True            Grup B  Responsable de vendes           0.0      NaN        NaN                   NaN                        14.7                1983-10-25


Con e DNI: 97954019T
- se puede eliminar  cualquiera de los dos registros poque son identicos.por ejemplo el 1002

In [128]:
df = df.drop(1002)

# Ejercicio 3

  Paso 1: Limpiar la columna numérica 'nombre de filis' rellenando con 0
  Si 'No Fills' es True o 'Fills' es False, asumimos que tiene 0 hijos de forma segura

In [ ]:

condicion_cero_hijos = (df["No Fills"] == True) | (df["Fills"] == False)
df.loc[condicion_cero_hijos, "Nombre_fills"] = df.loc[condicion_cero_hijos, "Nombre_fills"].fillna(0)

Paso 2: Crear la columna final unificada 'Fills'
Por defecto, tomamos el valor numérico que ya existe

In [111]:
df["Fills_ok"] = df["Nombre_fills"]

   Paso 3: Rectificar los NaN restantes cruzando las columnas booleanas
   Si 'Filis' es True pero el número de hijos es NaN, asumimos al menos 1 hijo (o valor indeterminado, Ej: 1)

In [112]:
df.loc[(df["Fills"] == True) & (df["Nombre_fills"].isna()), "Fills"] = 1

Si todo sigue siendo NaN (no hay info en ninguna columna), asumimos 0 por defecto para limpiar el dataset

In [113]:
df["Fills_ok"] = df["Fills_ok"].fillna(0)

Paso 4: Convertir a entero (int) para eliminar los decimales (.0) causados por los NaN

In [114]:
df["Fills_ok"] = df["Fills_ok"].astype(int)


Paso 5: Eliminar las 3 columnas que ya no necesitamos para dejar el dataset limpio

In [116]:
columnas_a_eliminar = ["Fills", "No Fills", "Nombre_fills"]
df = df.drop(columns=columnas_a_eliminar, errors="ignore")

In [131]:
display (df)

,Nom,Cognoms,DNI,País d'origen,Ciutat,Gènere,Salari mensual,Grup Professional,Carrec,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat,Fecha_Nacimiento_Validada,Fills_ok
0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,Dona,1469.00,Grup B,Cap de projecte,True,32108.0,25.0,10.1,1958-03-23,3
1,Marc,Muñoz,48840994W,Espanya,Alacant,Home,2718.00,Grup C,Senior analyst,True,19496.0,10.4,18.7,1960-11-08,0
2,Noa,Serra,14308421X,Espanya,Alacant,Home,1358.00,Grup A,Tecnic IT,NaN,NaN,NaN,16.7,1961-04-27,4
3,Pol,Gil,58586340F,Espanya,Sevilla,Home,1478.00,Grup B,Data Analyst,False,NaN,NaN,18.3,1985-10-13,2
4,David,Vila,82070937P,Espanya,Bilbao,Home,1284.00,Grup B,Administratiu,True,NaN,11.8,13.1,1965-11-29,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997,Jordi,Hernández,67755039Y,Espanya,Zaragoza,Home,1074.00,Grup A,Data Analyst,True,80000.0,12.0,12.0,1961-10-12,1
998,Chloé,Dubois,66354268T,Francia,Nice,Home,1954.00,Grup B,Data Analyst,NaN,NaN,NaN,11.7,1958-12-02,1
999,Adria,Vila Díaz,57511543T,Espanya,Palma,Home,2010.00,Grup C,Analista,False,NaN,NaN,-10.0,1991-12-31,1
1000,Sara,Serra,90585882A,Espanya,Tarragona,Dona,1073.00,Grup A,Administratiu,False,NaN,NaN,15.8,2002-05-28,0
